<a href="https://colab.research.google.com/github/SuyashPatil-max/CDC-x-Yhills/blob/main/model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score
from sklearn.model_selection import cross_val_score

!pip install xgboost
!pip install optuna


In [31]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [32]:
import tensorflow as tf
print('cuda' if tf.config.list_physical_devices('GPU') else 'cpu')


cuda


# Data Fetching


In [33]:
!unzip "/content/drive/MyDrive/CDC_Files_to_submit/images_df.zip"
!unzip "/content/drive/MyDrive/CDC_Files_to_submit/test_image_final.zip"

# paths of the dataset (might be different for u )

Archive:  /content/drive/MyDrive/CDC_Files_to_submit/images_df.zip
  inflating: images_df.csv           
Archive:  /content/drive/MyDrive/CDC_Files_to_submit/test_image_final.zip
  inflating: test_image.csv          


In [34]:
train = pd.read_csv('train_new.csv')
train.shape

(16209, 65)

# Tabular Data Prediction

In [35]:
X = train.iloc[:,:-1]
Y = train.iloc[:,-1]

print(X.shape ,Y.shape)

(16209, 64) (16209,)


In [36]:
from sklearn.preprocessing import PowerTransformer
trf = PowerTransformer(standardize = True)
Y = trf.fit_transform(Y.values.reshape(-1,1))

In [37]:
from xgboost import XGBRegressor
xgb = XGBRegressor()

In [38]:
xgb.fit(X ,Y)
cross_val_score(xgb , X,Y ,cv =10 ,scoring='r2').mean()

np.float64(0.8790048613039566)

In [39]:
import optuna

In [40]:
def objective(trial) :
  estimator = trial.suggest_int('n',5,100)
  depth = trial.suggest_int('d' ,5,20)
  lr = trial.suggest_float('lr',0.01,0.5)
  Lambda = trial.suggest_int('l',1,10)
  alpha = trial.suggest_int('al',0,10)
  child = trial.suggest_int('ch',1,5,2)
  subsample = trial.suggest_float('sub',0.7,1)
  col = trial.suggest_float('col',0.7 ,1 )

  model = XGBRegressor(n_estimators=estimator,max_depth=depth,learning_rate=lr,reg_lambda=Lambda,reg_alpha=alpha,
    min_child_weight=child,subsample=subsample,colsample_bytree=col,objective="reg:squarederror",
    random_state=42,n_jobs=-1)

  score = cross_val_score(model , X,Y ,cv =10 ,scoring='r2').mean()
  return score

In [41]:
study = optuna.create_study(direction ='minimize')
study.optimize(objective , n_trials=20)

[I 2026-01-07 16:36:29,256] A new study created in memory with name: no-name-cf5347ec-5965-47cb-9b56-062897e0ac4a
/tmp/ipython-input-473828992.py:7: FutureWarning: suggest_int() got {'step'} as positional arguments but they were expected to be given as keyword arguments.
Positional arguments ['self', 'name', 'low', 'high', 'step', 'log'] in suggest_int() have been deprecated since v3.5.0. They will be replaced with the corresponding keyword arguments in v5.0.0, so please use the keyword specification instead. See https://github.com/optuna/optuna/releases/tag/v3.5.0 for details.
  child = trial.suggest_int('ch',1,5,2)
[I 2026-01-07 16:36:45,971] Trial 0 finished with value: 0.8421251232057699 and parameters: {'n': 62, 'd': 20, 'lr': 0.4561034644522284, 'l': 1, 'al': 3, 'ch': 3, 'sub': 0.7691976349358105, 'col': 0.7113201452971384}. Best is trial 0 with value: 0.8421251232057699.
/tmp/ipython-input-473828992.py:7: FutureWarning: suggest_int() got {'step'} as positional arguments but they

In [42]:
op = study.trials_dataframe()
op.head()

,number,value,datetime_start,datetime_complete,duration,params_al,params_ch,params_col,params_d,params_l,params_lr,params_n,params_sub,state
0,0,0.842125,2026-01-07 16:36:29.257770,2026-01-07 16:36:45.971376,0 days 00:00:16.713606,3,3,0.711320,20,1,0.456103,62,0.769198,COMPLETE
1,1,0.658258,2026-01-07 16:36:45.972473,2026-01-07 16:36:48.064410,0 days 00:00:02.091937,2,5,0.841633,11,5,0.107737,8,0.730245,COMPLETE
2,2,0.873821,2026-01-07 16:36:48.065405,2026-01-07 16:36:56.273338,0 days 00:00:08.207933,0,5,0.772605,11,10,0.272200,38,0.798018,COMPLETE
3,3,0.881517,2026-01-07 16:36:56.274181,2026-01-07 16:37:02.065492,0 days 00:00:05.791311,8,1,0.958628,8,1,0.228259,69,0.825719,COMPLETE
4,4,0.862804,2026-01-07 16:37:02.067871,2026-01-07 16:37:10.800123,0 days 00:00:08.732252,4,5,0.953573,20,10,0.122564,25,0.779991,COMPLETE


In [43]:
j = 0
for i in op['number'] :
  if op['value'].max() == op['value'][i] :
    j =i
    print(op.iloc[i,:])
  else :
      pass

maxcol = op.iloc[j,:]

number                                       17
value                                  0.884931
datetime_start       2026-01-07 16:38:43.072297
datetime_complete    2026-01-07 16:38:46.715785
duration                 0 days 00:00:03.643488
params_al                                     1
params_ch                                     5
params_col                             0.822121
params_d                                      7
params_l                                      9
params_lr                              0.180237
params_n                                     53
params_sub                             0.998518
state                                  COMPLETE
Name: 17, dtype: object


In [44]:
maxcol

,17
number,17
value,0.884931
datetime_start,2026-01-07 16:38:43.072297
datetime_complete,2026-01-07 16:38:46.715785
duration,0 days 00:00:03.643488
params_al,1
params_ch,5
params_col,0.822121
params_d,7
params_l,9


In [45]:
model = XGBRegressor(n_estimators=maxcol[11],max_depth=maxcol[8],learning_rate=maxcol[10],reg_lambda=maxcol[9],reg_alpha=maxcol[5],
    min_child_weight=maxcol[6],subsample= maxcol[12],colsample_bytree=maxcol[7],objective="reg:squarederror",
    random_state=42,n_jobs=-1,device ='cuda')
model.fit(X ,Y)
cross_val_score(model , X,Y ,cv =10 ,scoring='r2').mean()

/tmp/ipython-input-1204714650.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  model = XGBRegressor(n_estimators=maxcol[11],max_depth=maxcol[8],learning_rate=maxcol[10],reg_lambda=maxcol[9],reg_alpha=maxcol[5],
/tmp/ipython-input-1204714650.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  min_child_weight=maxcol[6],subsample= maxcol[12],colsample_bytree=maxcol[7],objective="reg:squarederror",
/usr/local/lib/python3.12/dist-packages/xgboost/core.py:774: UserWarning: [16:39:11] WARNING: /workspace/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to hi

np.float64(0.8856547868740371)

# Tabular + Image

In [46]:
train_image = pd.read_csv('images_df.csv')
test_image = pd.read_csv('test_image.csv')

In [47]:
train_image.shape , test_image.shape

((16110, 2353), (5396, 2353))

In [48]:
drop_idx = np.random.choice(np.arange(X.shape[0]), size=X.shape[0] - train_image.shape[0], replace=False)
X_new = np.delete(X, drop_idx, axis=0)
Y_new = np.delete(Y, drop_idx, axis=0)

In [49]:
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.callbacks import EarlyStopping


In [50]:
from sklearn.model_selection import train_test_split
image_data_processed = train_image.drop(columns=['Unnamed: 0']).to_numpy()
X_img_train, X_img_test, X_tab_train, X_tab_test, y_train, y_test = train_test_split(image_data_processed, X_new, Y_new,test_size=0.2, random_state=42)
X_train_val_img =  X_img_train.reshape(-1,28,28,3)
train_image = image_data_processed.reshape(-1, 28, 28, 3)
X_img_test = X_img_test.reshape(-1, 28, 28, 3)

In [51]:
image_input = layers.Input(shape=(28, 28, 3), name="image_input")
x = layers.Resizing(224, 224)(image_input)

base_model = ResNet50(weights="imagenet",include_top=False,input_tensor=x)
base_model.trainable = False

x = layers.GlobalAveragePooling2D()(base_model.output)
x = layers.BatchNormalization()(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.3)(x)

image_features = layers.Dense(128, activation="relu")(x)


In [52]:
tabular_input = layers.Input(shape=(64,), name="tabular_input")

y = layers.BatchNormalization()(tabular_input)
y = layers.Dense(128, activation="relu")(y)
y = layers.Dropout(0.3)(y)
y = layers.Dense(64, activation="relu")(y)

tabular_features = layers.Dense(32, activation="relu")(y)


In [53]:
combined = layers.Concatenate()([image_features, tabular_features])

z = layers.Dense(128, activation="relu")(combined)
z = layers.Dropout(0.3)(z)
z = layers.Dense(64, activation="relu")(z)

output = layers.Dense(1, activation="linear", name="price")(z)


In [54]:
model = models.Model(
    inputs=[image_input, tabular_input],
    outputs=output
)

model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss="mse",
    metrics=["mae"]
)

model.summary()


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image_input         │ (None, 28, 28, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resizing_1          │ (None, 224, 224,  │          0 │ image_input[0][0] │
│ (Resizing)          │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ resizing_1[0][0]  │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c

 Total params: 24,201,185 (92.32 MB)

 Trainable params: 609,249 (2.32 MB)

 Non-trainable params: 23,591,936 (90.00 MB)

In [55]:
early_stop = EarlyStopping(monitor="val_loss",patience=5,restore_best_weights=True)
history = model.fit([X_train_val_img, X_tab_train],y_train,validation_split=0.2,epochs=10,batch_size=64,callbacks=[early_stop])

Epoch 1/10
162/162 ━━━━━━━━━━━━━━━━━━━━ 57s 259ms/step - loss: 0.6080 - mae: 0.6074 - val_loss: 45.9219 - val_mae: 6.7321
Epoch 2/10
162/162 ━━━━━━━━━━━━━━━━━━━━ 30s 182ms/step - loss: 0.3049 - mae: 0.4236 - val_loss: 7.0267 - val_mae: 2.5581
Epoch 3/10
162/162 ━━━━━━━━━━━━━━━━━━━━ 29s 181ms/step - loss: 0.2776 - mae: 0.4038 - val_loss: 1.1139 - val_mae: 0.9100
Epoch 4/10
162/162 ━━━━━━━━━━━━━━━━━━━━ 29s 181ms/step - loss: 0.2660 - mae: 0.3938 - val_loss: 0.2792 - val_mae: 0.3984
Epoch 5/10
162/162 ━━━━━━━━━━━━━━━━━━━━ 29s 181ms/step - loss: 0.2468 - mae: 0.3792 - val_loss: 0.2112 - val_mae: 0.3395
Epoch 6/10
162/162 ━━━━━━━━━━━━━━━━━━━━ 29s 181ms/step - loss: 0.2246 - mae: 0.3599 - val_loss: 0.2004 - val_mae: 0.3307
Epoch 7/10
162/162 ━━━━━━━━━━━━━━━━━━━━ 29s 181ms/step - loss: 0.2077 - mae: 0.3438 - val_loss: 0.1874 - val_mae: 0.3151
Epoch 8/10
162/162 ━━━━━━━━━━━━━━━━━━━━ 29s 180ms/step - loss: 0.1994 - mae: 0.3393 - val_loss: 0.1943 - val_mae: 0.3268
Epoch 9/10
162/162 ━━━━━━━━━━━━

In [56]:
from sklearn.metrics import r2_score
model.evaluate([X_img_test, X_tab_test], y_test)
y_pred = model.predict([X_img_test, X_tab_test])
r2_score_test = r2_score(y_test, y_pred)
print(r2_score_test)

101/101 ━━━━━━━━━━━━━━━━━━━━ 12s 99ms/step - loss: 0.1606 - mae: 0.2910
101/101 ━━━━━━━━━━━━━━━━━━━━ 15s 114ms/step
0.8298029941795921


In [57]:
from sklearn.metrics import r2_score
X_img_train = X_img_train.reshape(-1, 28, 28, 3)
model.evaluate([X_img_train, X_tab_train], y_train)
y_pred_train = model.predict([X_img_train, X_tab_train])
train_r2_score = r2_score(y_train, y_pred_train)
print(train_r2_score)

403/403 ━━━━━━━━━━━━━━━━━━━━ 35s 86ms/step - loss: 0.1435 - mae: 0.2773
403/403 ━━━━━━━━━━━━━━━━━━━━ 33s 81ms/step
0.8513417881798798


In [58]:
test_data = pd.read_csv('test_new.csv')
test_data.shape , test_image.shape

((5404, 63), (5396, 2353))

In [59]:
test_data.head()

,Unnamed: 0,id,sqft_living,sqft_lot,sqft_above,sqft_living15,sqft_lot15,year,month_sin,month_cos,...,grade_other,yr_renovated_0,yr_renovated_other,sqft_basement_0,sqft_basement_400,sqft_basement_500,sqft_basement_600,sqft_basement_700,sqft_basement_800,sqft_basement_other
0,0,2591820310,0.166806,0.199326,0.591864,0.755685,-0.000315,-4.463097e-14,-1.224746,0.974179,...,0,1,0,1,0,0,0,0,0,0
1,1,7974200820,1.009472,-0.254074,0.308057,0.730493,-0.353140,-4.463097e-14,-1.224746,-0.435512,...,0,1,0,0,0,0,0,0,0,1
2,2,7701450110,1.697622,0.559292,1.899235,2.009726,0.435429,-4.463097e-14,-1.224746,-0.435512,...,0,1,0,1,0,0,0,0,0,0
3,3,9522300010,2.034593,1.129022,1.945594,2.009726,1.266848,9.242607e-14,1.413556,0.337955,...,1,1,0,1,0,0,0,0,0,0
4,4,9510861140,0.682605,-0.587696,1.060206,0.573921,-1.019664,-4.463097e-14,-0.697348,-1.083843,...,0,1,0,1,0,0,0,0,0,0


In [60]:
drop_idx = np.random.choice(np.arange(test_data.shape[0]), size=test_data.shape[0] - test_image.shape[0], replace=False)
test_new = np.delete(test_data, drop_idx, axis=0)

In [61]:
test_new.shape

(5396, 63)

In [62]:
test_cols = test_data.columns
test_cols

Index(['Unnamed: 0', 'id', 'sqft_living', 'sqft_lot', 'sqft_above',
       'sqft_living15', 'sqft_lot15', 'year', 'month_sin', 'month_cos',
       'dow_sin', 'dow_cos', 'floors', 'yr_built', 'zipcode', 'lat', 'long',
       'bedrooms_1', 'bedrooms_2', 'bedrooms_3', 'bedrooms_4', 'bedrooms_5',
       'bedrooms_6', 'bedrooms_other', 'bathrooms_1.0', 'bathrooms_1.5',
       'bathrooms_1.75', 'bathrooms_2.0', 'bathrooms_2.25', 'bathrooms_2.5',
       'bathrooms_2.75', 'bathrooms_3.0', 'bathrooms_3.25', 'bathrooms_3.5',
       'bathrooms_other', 'waterfront_0', 'waterfront_1', 'view_0', 'view_1',
       'view_2', 'view_3', 'view_4', 'condition_1', 'condition_2',
       'condition_3', 'condition_4', 'condition_5', 'grade_10', 'grade_11',
       'grade_6', 'grade_7', 'grade_8', 'grade_9', 'grade_other',
       'yr_renovated_0', 'yr_renovated_other', 'sqft_basement_0',
       'sqft_basement_400', 'sqft_basement_500', 'sqft_basement_600',
       'sqft_basement_700', 'sqft_basement_800', 'sqft_b

In [63]:
test_new = pd.DataFrame(test_new ,columns = test_cols)

In [64]:
test_new.head()

,Unnamed: 0,id,sqft_living,sqft_lot,sqft_above,sqft_living15,sqft_lot15,year,month_sin,month_cos,...,grade_other,yr_renovated_0,yr_renovated_other,sqft_basement_0,sqft_basement_400,sqft_basement_500,sqft_basement_600,sqft_basement_700,sqft_basement_800,sqft_basement_other
0,0.0,2.591820e+09,0.166806,0.199326,0.591864,0.755685,-0.000315,-4.463097e-14,-1.224746,0.974179,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1.0,7.974201e+09,1.009472,-0.254074,0.308057,0.730493,-0.353140,-4.463097e-14,-1.224746,-0.435512,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,2.0,7.701450e+09,1.697622,0.559292,1.899235,2.009726,0.435429,-4.463097e-14,-1.224746,-0.435512,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
3,3.0,9.522300e+09,2.034593,1.129022,1.945594,2.009726,1.266848,9.242607e-14,1.413556,0.337955,...,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4,4.0,9.510861e+09,0.682605,-0.587696,1.060206,0.573921,-1.019664,-4.463097e-14,-0.697348,-1.083843,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


In [65]:
test_new = test_new.drop('Unnamed: 0',axis =1 )

In [66]:
test_id = test_new['id']
test_new = test_new.drop(['id'],axis = 1)

In [67]:
test_new.head()

,sqft_living,sqft_lot,sqft_above,sqft_living15,sqft_lot15,year,month_sin,month_cos,dow_sin,dow_cos,...,grade_other,yr_renovated_0,yr_renovated_other,sqft_basement_0,sqft_basement_400,sqft_basement_500,sqft_basement_600,sqft_basement_700,sqft_basement_800,sqft_basement_other
0,0.166806,0.199326,0.591864,0.755685,-0.000315,-4.463097e-14,-1.224746,0.974179,-0.769838,1.329658,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1.009472,-0.254074,0.308057,0.730493,-0.353140,-4.463097e-14,-1.224746,-0.435512,0.021347,-1.153011,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,1.697622,0.559292,1.899235,2.009726,0.435429,-4.463097e-14,-1.224746,-0.435512,-1.387319,-1.153011,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2.034593,1.129022,1.945594,2.009726,1.266848,9.242607e-14,1.413556,0.337955,0.782134,0.889491,...,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.682605,-0.587696,1.060206,0.573921,-1.019664,-4.463097e-14,-0.697348,-1.083843,-0.769838,1.329658,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


In [68]:
X_img_for_pred = test_image.drop(columns=['Unnamed: 0']).to_numpy().reshape(-1, 28, 28, 3)

X_tab_cols = X.columns.tolist()

test_data_filtered_rows = test_data.drop(test_data.index[drop_idx])

X_tab_for_pred_df = test_data_filtered_rows.reindex(columns=X_tab_cols, fill_value=0)
X_tab_for_pred = X_tab_for_pred_df.to_numpy()

y_pred = model.predict([X_img_for_pred, X_tab_for_pred])
y_pred = y_pred.reshape(-1)

169/169 ━━━━━━━━━━━━━━━━━━━━ 17s 98ms/step


In [69]:
y_pred.shape , test_id.shape

((5396,), (5396,))

In [70]:
test_id

,id
0,2.591820e+09
1,7.974201e+09
2,7.701450e+09
3,9.522300e+09
4,9.510861e+09
...,...
5391,7.732500e+09
5392,3.856904e+09
5393,2.557000e+09
5394,4.386700e+09


In [71]:
y_pred_real = trf.inverse_transform(y_pred.reshape(-1, 1)).ravel()
y_pred_real.shape

(5396,)

In [72]:
submission = pd.DataFrame({"id":test_id,"price": y_pred_real})
submission.to_csv("submission.csv", index=False)

print("submission.csv saved")
print(submission.head())

submission.csv saved
             id         price
0  2.591820e+09  3.858122e+05
1  7.974201e+09  7.635859e+05
2  7.701450e+09  1.177037e+06
3  9.522300e+09  1.610122e+06
4  9.510861e+09  6.377921e+05
